# Transformer From Scratch in PyTorch

This notebook keeps only the Transformer-building path:

- data preprocessing
- Word2Vec embeddings
- token IDs for targets
- padding and batching
- single-head self-attention
- efficient multi-head self-attention
- positional encoding
- encoder block
- masked multi-head self-attention
- multi-head cross-attention
- decoder block
- 6-layer encoder + 6-layer decoder training pipeline

The target right shift is performed once in the training loop, not inside each decoder block.


In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split


## 1. Data Processing


In [ ]:
x_words=df['English words/sentences'].fillna('').apply(
    lambda sentence:sentence.lower().split()
)

y_words=df['French words/sentences'].fillna('').apply(
    lambda sentence:sentence.lower().split()
)

x_train_words,x_test_words,y_train_words,y_test_words=train_test_split(
    x_words,
    y_words,
    test_size=0.2,
    random_state=0
)

x_train_words=list(x_train_words)
x_test_words=list(x_test_words)
y_train_words=list(y_train_words)
y_test_words=list(y_test_words)


## 2. Train Word2Vec on Training Data Only


In [ ]:
modelx=Word2Vec(
    x_train_words,
    vector_size=512,
    window=2,
    min_count=1,
    sg=1
)

modely=Word2Vec(
    y_train_words,
    vector_size=512,
    window=2,
    min_count=1,
    sg=1
)


## 3. Build French Target IDs

The decoder input uses French embeddings, but `CrossEntropyLoss` needs integer target IDs.
A separate `<pad>` ID is reserved outside the Word2Vec vocabulary.


In [ ]:
vocab_size=len(modely.wv.index_to_key)
pad_id=vocab_size
output_vocab_size=vocab_size+1

y_train_ids=[]

for sentence in y_train_words:
    temp=[]

    for word in sentence:
        temp.append(modely.wv.key_to_index[word])

    y_train_ids.append(temp)


## 4. Convert Training Words to 512-D Embeddings


In [ ]:
x_train=[]

for sentence in x_train_words:
    temp=[]

    for word in sentence:
        temp.append(modelx.wv[word])

    x_train.append(temp)


y_train=[]

for sentence in y_train_words:
    temp=[]

    for word in sentence:
        temp.append(modely.wv[word])

    y_train.append(temp)


## 5. Pad Encoder Inputs, Decoder Inputs, and Target IDs


In [ ]:
max_src_len=max(len(sentence) for sentence in x_train)
max_tgt_len=max(len(sentence) for sentence in y_train)

x_padded=[]

for sentence in x_train:
    sentence=torch.from_numpy(np.array(sentence)).float()
    pad_len=max_src_len-len(sentence)

    if pad_len>0:
        padding=torch.zeros(pad_len,512)
        sentence=torch.cat((sentence,padding),dim=0)

    x_padded.append(sentence)

x_padded=torch.stack(x_padded)


y_padded=[]

for sentence in y_train:
    sentence=torch.from_numpy(np.array(sentence)).float()
    pad_len=max_tgt_len-len(sentence)

    if pad_len>0:
        padding=torch.zeros(pad_len,512)
        sentence=torch.cat((sentence,padding),dim=0)

    y_padded.append(sentence)

y_padded=torch.stack(y_padded)


y_ids_padded=[]

for sentence in y_train_ids:
    pad_len=max_tgt_len-len(sentence)
    sentence=sentence+[pad_id]*pad_len
    y_ids_padded.append(sentence)

y_ids_padded=torch.tensor(y_ids_padded,dtype=torch.long)


## 6. Make Sequential Batches of 5


In [ ]:
batch_size=5

x_batches=[]
y_batches=[]
y_target_batches=[]

for i in range(0,len(x_padded),batch_size):
    xbatch=x_padded[i:i+batch_size]
    ybatch=y_padded[i:i+batch_size]
    targetbatch=y_ids_padded[i:i+batch_size]

    if len(xbatch)==batch_size:
        x_batches.append(xbatch)
        y_batches.append(ybatch)
        y_target_batches.append(targetbatch)

x_batches=torch.stack(x_batches)
y_batches=torch.stack(y_batches)
y_target_batches=torch.stack(y_target_batches)

print("x_batches:",x_batches.shape)
print("y_batches:",y_batches.shape)
print("y_target_batches:",y_target_batches.shape)


## 7. Single-Head Self-Attention


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self,embed_size):
        super().__init__()

        self.embed_size=embed_size

        self.Q=nn.Linear(embed_size,embed_size)
        self.K=nn.Linear(embed_size,embed_size)
        self.V=nn.Linear(embed_size,embed_size)

    def forward(self,embedding):
        q=self.Q(embedding)
        k=self.K(embedding)
        v=self.V(embedding)

        scores=q@k.transpose(-2,-1)
        scores=scores/math.sqrt(self.embed_size)

        attention=torch.softmax(scores,dim=-1)

        output=attention@v

        return output


## 8. Efficient Multi-Head Self-Attention


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embed_size,heads):
        super().__init__()

        self.embed_size=embed_size
        self.heads=heads
        self.head_dim=embed_size//heads

        assert embed_size%heads==0

        self.query=nn.Linear(embed_size,embed_size)
        self.key=nn.Linear(embed_size,embed_size)
        self.value=nn.Linear(embed_size,embed_size)

        self.out=nn.Linear(embed_size,embed_size)

    def forward(self,x):
        batch_size,seq_len,embed_size=x.shape

        Q=self.query(x)
        K=self.key(x)
        V=self.value(x)

        Q=Q.reshape(batch_size,seq_len,self.heads,self.head_dim)
        K=K.reshape(batch_size,seq_len,self.heads,self.head_dim)
        V=V.reshape(batch_size,seq_len,self.heads,self.head_dim)

        Q=Q.transpose(1,2)
        K=K.transpose(1,2)
        V=V.transpose(1,2)

        scores=Q@K.transpose(-2,-1)
        scores=scores/math.sqrt(self.head_dim)

        attention=torch.softmax(scores,dim=-1)

        output=attention@V
        output=output.transpose(1,2)
        output=output.reshape(batch_size,seq_len,embed_size)

        output=self.out(output)

        return output


## 9. Positional Encoding


In [ ]:
class PositionalEncoder(nn.Module):
    def __init__(self,embed_size,max_length):
        super().__init__()

        self.embed_size=embed_size
        self.max_length=max_length

        pe=torch.zeros(max_length,embed_size)

        for pos in range(max_length):
            for i in range(0,embed_size,2):
                pe[pos][i]=math.sin(pos/10000**(i/embed_size))

                if i+1<embed_size:
                    pe[pos][i+1]=math.cos(pos/10000**(i/embed_size))

        self.register_buffer("pe",pe)

    def forward(self,x):
        return x+self.pe[:x.shape[1]]


## 10. Encoder Block

Positional encoding is not applied inside the block. It will be added once before the encoder stack.


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self,embed_size,heads):
        super().__init__()

        self.MHA=MultiHeadAttention(embed_size,heads)

        self.norm1=nn.LayerNorm(embed_size)
        self.norm2=nn.LayerNorm(embed_size)

        self.FeedForwardLayer=nn.Sequential(
            nn.Linear(embed_size,4*embed_size),
            nn.ReLU(),
            nn.Linear(4*embed_size,embed_size)
        )

    def forward(self,x):
        attention=self.MHA(x)

        out1=attention+x
        out2=self.norm1(out1)

        ff=self.FeedForwardLayer(out2)

        out3=ff+out2
        out4=self.norm2(out3)

        return out4


## 11. Masked Multi-Head Self-Attention


In [ ]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self,embed_size,heads):
        super().__init__()

        self.embed_size=embed_size
        self.heads=heads
        self.head_dim=embed_size//heads

        assert embed_size%heads==0

        self.query=nn.Linear(embed_size,embed_size)
        self.key=nn.Linear(embed_size,embed_size)
        self.value=nn.Linear(embed_size,embed_size)

        self.out=nn.Linear(embed_size,embed_size)

    def forward(self,x):
        batch_size,seq_len,embed_size=x.shape

        Q=self.query(x)
        K=self.key(x)
        V=self.value(x)

        Q=Q.reshape(batch_size,seq_len,self.heads,self.head_dim)
        K=K.reshape(batch_size,seq_len,self.heads,self.head_dim)
        V=V.reshape(batch_size,seq_len,self.heads,self.head_dim)

        Q=Q.transpose(1,2)
        K=K.transpose(1,2)
        V=V.transpose(1,2)

        scores=Q@K.transpose(-2,-1)
        scores=scores/math.sqrt(self.head_dim)

        matrix=torch.zeros((seq_len,seq_len),device=x.device)

        for i in range(seq_len):
            for j in range(seq_len):
                if i<j:
                    matrix[i][j]=-torch.inf

        scores=scores+matrix

        attention=torch.softmax(scores,dim=-1)

        output=attention@V
        output=output.transpose(1,2)
        output=output.reshape(batch_size,seq_len,embed_size)

        output=self.out(output)

        return output


## 12. Multi-Head Cross-Attention


In [ ]:
class CrossMultiHeadAttention(nn.Module):
    def __init__(self,embed_size,heads):
        super().__init__()

        self.embed_size=embed_size
        self.heads=heads
        self.head_dim=embed_size//heads

        assert embed_size%heads==0

        self.query=nn.Linear(embed_size,embed_size)
        self.key=nn.Linear(embed_size,embed_size)
        self.value=nn.Linear(embed_size,embed_size)

        self.out=nn.Linear(embed_size,embed_size)

    def forward(self,outputemb,encemb):
        batch_size=outputemb.shape[0]

        out_seq_len=outputemb.shape[1]
        enc_seq_len=encemb.shape[1]

        Q=self.query(outputemb)
        K=self.key(encemb)
        V=self.value(encemb)

        Q=Q.reshape(batch_size,out_seq_len,self.heads,self.head_dim)
        K=K.reshape(batch_size,enc_seq_len,self.heads,self.head_dim)
        V=V.reshape(batch_size,enc_seq_len,self.heads,self.head_dim)

        Q=Q.transpose(1,2)
        K=K.transpose(1,2)
        V=V.transpose(1,2)

        scores=Q@K.transpose(-2,-1)
        scores=scores/math.sqrt(self.head_dim)

        attention=torch.softmax(scores,dim=-1)

        output=attention@V
        output=output.transpose(1,2)
        output=output.reshape(batch_size,out_seq_len,self.embed_size)

        output=self.out(output)

        return output


## 13. Decoder Block

The target sequence is shifted once before the decoder stack. Positional encoding is also added once before the stack.


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self,embed_size,heads):
        super().__init__()

        self.MHA=MaskedMultiHeadAttention(embed_size,heads)
        self.cross=CrossMultiHeadAttention(embed_size,heads)

        self.norm1=nn.LayerNorm(embed_size)
        self.norm2=nn.LayerNorm(embed_size)
        self.norm3=nn.LayerNorm(embed_size)

        self.FeedForwardLayer=nn.Sequential(
            nn.Linear(embed_size,4*embed_size),
            nn.ReLU(),
            nn.Linear(4*embed_size,embed_size)
        )

    def forward(self,x,encoderout):
        attention=self.MHA(x)

        out1=attention+x
        out2=self.norm1(out1)

        crossattention=self.cross(out2,encoderout)

        out3=crossattention+out2
        out4=self.norm2(out3)

        ff=self.FeedForwardLayer(out4)

        out5=ff+out4
        out6=self.norm3(out5)

        return out6


## 14. Build the 6-Encoder / 6-Decoder Transformer


In [ ]:
embed_size=512
heads=8

encoder_pe=PositionalEncoder(embed_size,max_src_len)
decoder_pe=PositionalEncoder(embed_size,max_tgt_len)

e1=EncoderBlock(embed_size,heads)
e2=EncoderBlock(embed_size,heads)
e3=EncoderBlock(embed_size,heads)
e4=EncoderBlock(embed_size,heads)
e5=EncoderBlock(embed_size,heads)
e6=EncoderBlock(embed_size,heads)

d1=DecoderBlock(embed_size,heads)
d2=DecoderBlock(embed_size,heads)
d3=DecoderBlock(embed_size,heads)
d4=DecoderBlock(embed_size,heads)
d5=DecoderBlock(embed_size,heads)
d6=DecoderBlock(embed_size,heads)

start=nn.Parameter(torch.randn(1,embed_size))

output_layer=nn.Linear(embed_size,output_vocab_size)

criterion=nn.CrossEntropyLoss(ignore_index=pad_id)


## 15. Optimizer


In [ ]:
params=[]

for model_part in [
    encoder_pe,
    decoder_pe,
    e1,e2,e3,e4,e5,e6,
    d1,d2,d3,d4,d5,d6,
    output_layer
]:
    params+=list(model_part.parameters())

params.append(start)

optimizer=torch.optim.Adam(params,lr=0.001)


## 16. Training Loop


In [ ]:
epochs=50

for epoch in range(epochs):
    total_loss=0

    for j in range(len(x_batches)):
        xbatch=x_batches[j]
        ybatch=y_batches[j]
        y_target=y_target_batches[j]

        optimizer.zero_grad()

        xbatch=encoder_pe(xbatch)

        out1=e1(xbatch)
        out2=e2(out1)
        out3=e3(out2)
        out4=e4(out3)
        out5=e5(out4)
        out6=e6(out5)

        shifted=[]

        for i in range(ybatch.shape[0]):
            shifted.append(
                torch.cat(
                    (
                        start,
                        ybatch[i][:-1]
                    ),
                    dim=0
                )
            )

        shifted=torch.stack(shifted)
        shifted=decoder_pe(shifted)

        out7=d1(shifted,out6)
        out8=d2(out7,out6)
        out9=d3(out8,out6)
        out10=d4(out9,out6)
        out11=d5(out10,out6)
        out12=d6(out11,out6)

        logits=output_layer(out12)

        loss=criterion(
            logits.reshape(-1,output_vocab_size),
            y_target.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss+=loss.item()

    avg_loss=total_loss/len(x_batches)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {avg_loss:.4f}"
    )


## 17. Output Probabilities and Predicted Token IDs


In [ ]:
probabilities=torch.softmax(logits,dim=-1)
predicted_ids=torch.argmax(probabilities,dim=-1)

print("Probabilities shape:",probabilities.shape)
print("Predicted IDs shape:",predicted_ids.shape)
